# RunnableParallel

This notebook requires the `grandalf` dependency

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [5]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books.
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers.
    Answer only by listing the projects.
    '''
)

In [6]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1, max_completion_tokens=10000)

In [8]:
string_parser = StrOutputParser()

In [10]:
chain_books = chat_template_books | chat | string_parser
chain_projects = chat_template_projects | chat | string_parser

In [11]:
chain_parallel = RunnableParallel({'books': chain_books,'projects': chain_projects})

In [12]:
print(chain_parallel.invoke({'programming language': 'Python'}))

{'books': '- *Effective Python: 125 Specific Ways to Write Better Python* — Brett Slatkin\n- *Fluent Python* (2nd ed.) — Luciano Ramalho\n- *Python Cookbook* (3rd ed.) — David Beazley and Brian K. Jones', 'projects': '- **Personal Finance Tracker:** Build an app that imports bank transactions, categorizes spending, and visualizes monthly budgets.\n- **Collaborative Markdown Editor:** Create a web app with live editing, version history, and user authentication.\n- **Smart Home Dashboard:** Monitor and control simulated devices, with scheduled actions and energy-use charts.'}


In [15]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

# Timing comparisons

In [16]:
%%time
chain_books.invoke({'programming language': 'Python'})

CPU times: user 10.4 ms, sys: 1.15 ms, total: 11.5 ms
Wall time: 3 s


'- *Fluent Python*, 2nd Edition — Luciano Ramalho\n- *Effective Python*, 2nd Edition — Brett Slatkin\n- *Python Cookbook*, 3rd Edition — David Beazley and Brian K. Jones'

In [17]:
%%time
chain_projects.invoke({'programming language': 'Python'})

CPU times: user 7.31 ms, sys: 1.98 ms, total: 9.3 ms
Wall time: 1.89 s


'1. **Personal Finance Dashboard** — Build an app that imports bank transactions, categorizes spending, and visualizes budgets and trends.\n2. **Smart Home Energy Monitor** — Collect simulated or real sensor data, detect usage patterns, and display energy consumption over time.\n3. **Collaborative Markdown Editor** — Create a web app where users can edit documents together, track revisions, and leave comments.'

In [18]:
%%time
chain_parallel.invoke({'programming language': 'Python'})

CPU times: user 18.4 ms, sys: 3.96 ms, total: 22.4 ms
Wall time: 3.86 s


{'books': '- *Fluent Python*, 2nd Edition — Luciano Ramalho\n- *Effective Python*, 3rd Edition — Brett Slatkin\n- *Python Cookbook*, 3rd Edition — David Beazley and Brian K. Jones',
 'projects': '- **Personal Finance Dashboard:** Build a Python app that imports bank transactions, categorizes spending, and visualizes budgets and trends.\n- **Local Event Finder:** Create a tool that collects event listings from public APIs, filters them by interests and location, and sends a weekly digest.\n- **Document Search Engine:** Build a program that indexes a folder of PDFs and text files, then finds relevant passages using full-text search and embeddings.'}